# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIRˆ2 dataset using the `mlcroissant` library, following the Croissant schema and referencing all data entities via their `@id` fields.

### Dataset Source
The dataset is described by a Croissant schema available at:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Initialize the dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")
print(f"Identifier: {metadata.identifier}\nVersion: {metadata.version}\nLicense: {metadata.license}")

## 2. Data Overview
Review all available record sets, their fields, and Croissant `@id` values as required for referencing in subsequent sections.

**Note:** All references below use the canonical `@id` for each dataset component.

In [ ]:
# List all record sets available via their @id
print("Available Record Sets:")
recordset_ids = []
for rs in metadata.record_sets:
    print(f"- @id: {rs.id}, name: {getattr(rs, 'name', '<no name>')}")
    recordset_ids.append(rs.id)

# For each record set, print its fields and columns by @id
for rs in metadata.record_sets:
    print(f"\nRecord Set @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {getattr(field, 'name', '<no name>')}, dataType: {getattr(field, 'data_type', None)}")
        if hasattr(field, 'columns'):
            for col in field.columns:
                print(f"      Column @id: {col.id}, name: {getattr(col, 'name', '<no name>')}")

## 3. Data Extraction
Load the data from each record set into a DataFrame, referencing only the `@id` of the record set and its fields.

In [ ]:
# Build a dictionary of DataFrames for each record set using its @id
dataframes = {}

for record_set_id in recordset_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[record_set_id])} records for RecordSet: {record_set_id}")

# For demonstration, pick the first record set for the next steps
if recordset_ids:
    demo_recordset_id = recordset_ids[0]
    print(f"\nColumns in DataFrame for RecordSet {demo_recordset_id}:")
    print(dataframes[demo_recordset_id].columns.tolist())
    display(dataframes[demo_recordset_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing techniques, referencing fields/columns by `@id` only. We will filter, normalize, and group by example fields as found above.

In [ ]:
# For EDA, pick a numeric field and a grouping field from the fields of the first record set
demo_fields = [field.id for field in metadata.record_sets[0].fields]
print(f"Available fields (@id) in demo RecordSet: {demo_fields}")

# Example: select first numeric field (by type) and first string/categorical field as grouping
numeric_field_id = None
group_field_id = None
for field in metadata.record_sets[0].fields:
    if getattr(field, 'data_type', None) in ['schema:Float', 'schema:Integer', 'schema:Number'] and not numeric_field_id:
        numeric_field_id = field.id
    if getattr(field, 'data_type', None) == 'schema:Text' and not group_field_id:
        group_field_id = field.id

print(f"Numeric field selected (@id): {numeric_field_id}")
print(f"Group-by field selected (@id): {group_field_id}")

# Use the DataFrame for EDA
df = dataframes[demo_recordset_id]

# Check for missing values in the numeric field
if numeric_field_id and numeric_field_id in df.columns:
    numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = numeric_series.quantile(0.75)  # Example: filter above 75th percentile
    filtered_df = df[numeric_series > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (75th percentile): {len(filtered_df)}")

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (numeric_series - numeric_series.mean()) / numeric_series.std()
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Grouping
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} by {group_field_id} after filtering:")
        display(grouped_df.head())

## 5. Visualization
Visualize a histogram of the selected numeric field for the record set, and (optionally) a boxplot grouped by a categorical field, all referenced by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
    plt.figure(figsize=(8, 5))
    sns.histplot(numeric_series.dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field_id], y=numeric_series)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we've loaded and explored the FAIRˆ2 dataset using the `mlcroissant` library. All entities, including record sets, fields, and columns, were referenced exclusively via their Croissant `@id` to ensure reproducibility and schema clarity. Analysis steps demonstrated filtering, normalization, grouping, and visualization techniques that can be extended to richer analyses. For additional record sets or variables, follow the demonstrated `@id` referencing pattern for robust FAIR-compliant workflows.